# Chiller Load Forecasting — Exploratory Data Analysis

This notebook explores the ASHRAE Great Energy Predictor III dataset, focusing on **chilled water** (meter type 1) for cooling load prediction.

**Goal**: Understand data distribution, temporal patterns, missing values, and relationships between features and cooling load.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 50)

from src.data.loader import load_chilled_water_data
from src.data.preprocessing import clean_data, data_quality_report
from src.data.features import build_features, get_feature_columns

## 1. Load & Inspect Raw Data

In [ ]:
df = load_chilled_water_data()
print(f"Shape: {df.shape}")
print(f"Buildings: {df['building_id'].nunique()}")
print(f"Sites: {df['site_id'].nunique()}")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
df.head()

In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0").sort_values("pct", ascending=False)

## 2. Target Distribution — Chilled Water Consumption

In [ ]:
fig = px.histogram(df, x="meter_reading", nbins=100, log_y=True,
                    title="Chilled Water Meter Reading Distribution (log scale)",
                    labels={"meter_reading": "kWh"})
fig.show()

In [ ]:
# Pick a representative building for time-series plots
sample_bid = df.groupby("building_id")["meter_reading"].count().idxmax()
sample = df[df["building_id"] == sample_bid].copy()
print(f"Sample building: {sample_bid}  ({len(sample)} rows)")

fig = px.line(sample, x="timestamp", y="meter_reading",
              title=f"Building {sample_bid} — Chilled Water Over Time")
fig.show()

## 3. Temporal Patterns

Cooling load follows strong daily and seasonal cycles. Business hours and summer months drive higher consumption.

In [ ]:
sample["hour"] = sample["timestamp"].dt.hour
sample["month"] = sample["timestamp"].dt.month

fig = make_subplots(rows=1, cols=2, subplot_titles=["Hourly Pattern", "Monthly Pattern"])
hourly = sample.groupby("hour")["meter_reading"].mean()
monthly = sample.groupby("month")["meter_reading"].mean()

fig.add_trace(go.Bar(x=hourly.index, y=hourly.values, name="Hour"), row=1, col=1)
fig.add_trace(go.Bar(x=monthly.index, y=monthly.values, name="Month"), row=1, col=2)
fig.update_layout(title=f"Building {sample_bid} — Temporal Patterns", showlegend=False)
fig.show()

## 4. Weather vs Cooling Load

Outdoor temperature is the primary driver of chiller cooling load. We expect a strong positive correlation above the base temperature (~18°C).

In [ ]:
fig = px.scatter(sample.dropna(subset=["air_temperature", "meter_reading"]),
                  x="air_temperature", y="meter_reading",
                  opacity=0.3, trendline="lowess",
                  title=f"Building {sample_bid} — Air Temperature vs Cooling Load",
                  labels={"air_temperature": "Air Temperature (°C)", "meter_reading": "Load (kWh)"})
fig.show()

## 5. Data Cleaning & Feature Engineering

In [ ]:
df_clean = clean_data(df)
report = data_quality_report(df_clean)
print("Data quality summary (top 10 by missing %):")
report.sort_values("missing_pct", ascending=False).head(10)

In [ ]:
df_feat = build_features(df_clean)
feature_cols = get_feature_columns(df_feat)
print(f"Total features: {len(feature_cols)}")
print("Feature columns:", feature_cols[:15], "...")

## 6. Feature Correlation with Target

Understanding which features have the strongest relationship with cooling load helps validate our feature engineering choices.

In [ ]:
corr = df_feat[feature_cols + ["meter_reading"]].corr()["meter_reading"].drop("meter_reading")
top_corr = corr.abs().sort_values(ascending=False).head(15)

fig = px.bar(x=top_corr.values, y=top_corr.index, orientation="h",
             title="Top 15 Features by Absolute Correlation with Cooling Load",
             labels={"x": "|Correlation|", "y": "Feature"})
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()